### Part one : load data and split it

In [34]:
import pandas as pd 
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.decomposition import PCA

In [35]:
path = "data/penguins_with_splits.csv"
data = pd.read_csv(path)
row , col = data.shape
print(f"Rows: {row}, Columns: {col}")
data.head()

Rows: 344, Columns: 9


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year,split
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007,train
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007,val
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007,train
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007,train
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007,train


In [36]:
test = data[data['split'] == 'test']
val = data[data['split'] == 'val']
train = data[data['split'] == 'train']

print("Train:", train.shape)
print("Validation:", val.shape)
print("Test:", test.shape)

Train: (241, 9)
Validation: (69, 9)
Test: (34, 9)


### Part tow : Build a Preprocessing Pipeline

In [37]:
numeric_features = [
  "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g",
    "year"
]

categorical_features = ["island", "sex"]

def add_bill_ratio(X):
    ratio = X[:, 0] / X[:, 1]
    return np.column_stack((X, ratio))


def ratio_feature_names(transformer, input_features):
    return list(input_features) + ["bill_shape_ratio"]

## build pipeline

In [38]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("ratio", FunctionTransformer(
        add_bill_ratio,
        feature_names_out=ratio_feature_names
    )),
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95))
])

categorical_pipeline = Pipeline([
  ("imputer",SimpleImputer(strategy="most_frequent")),
  ("encoder", OneHotEncoder(
    handle_unknown="ignore" ,
    sparse_output=False
  ))
])


preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, ["island"])
])

pipeline = Pipeline([
    ("preprocessor", preprocessor)
])

### Part 3 — Apply the Pipeline

In [39]:
input_features = numeric_features + ["island"]

x_train = train[input_features]
x_val = val[input_features]
x_test = test[input_features]

In [40]:
train_processed = pipeline.fit_transform(x_train)
test_processed = pipeline.transform(x_test)
val_processed = pipeline.transform(x_val)

In [41]:
print("Train:", train_processed.shape)
print("Validation:", val_processed.shape)
print("Test:", test_processed.shape)

Train: (241, 7)
Validation: (69, 7)
Test: (34, 7)


In [42]:
print("Missing values:", np.isnan(train_processed).sum())

pca = pipeline.named_steps["preprocessor"] \
    .named_transformers_["num"] \
    .named_steps["pca"]

print("PCA Components:", pca.n_components_)
print("Explained Variance:", pca.explained_variance_ratio_.sum())

Missing values: 0
PCA Components: 4
Explained Variance: 0.9812771158399247


### Part 4: Save the processed dataset

In [43]:
feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()

train_df = pd.DataFrame(
    train_processed,
    columns=feature_names,
    index=x_train.index
)

val_df = pd.DataFrame(
    val_processed,
    columns=feature_names,
    index=x_val.index
)

test_df = pd.DataFrame(
    test_processed,
    columns=feature_names,
    index=x_test.index
)

In [44]:
processed_features = pd.concat([
    train_df,
    val_df,
    test_df
])

processed_features = processed_features.sort_index()

In [45]:
metadata = data.loc[
    processed_features.index,
    ["species", "split"]
].copy()

processed_data = pd.concat([
    processed_features,
    metadata
], axis=1)

processed_data.insert(
    0,
    "original_index",
    processed_data.index
)

processed_data = processed_data.reset_index(drop=True)

In [46]:
processed_data.to_csv("data/penguins_processed.csv", index=False)

print("File saved successfully!")
print(processed_data.head())

File saved successfully!
   original_index  num__pca0  num__pca1  num__pca2  num__pca3  \
0               0  -2.176416  -1.166923  -0.050770   0.064370   
1               1  -1.503375  -1.168211  -0.538871  -0.059486   
2               2  -1.584433  -1.048221  -0.266854  -0.244730   
3               3  -0.260799  -1.189871   0.159971  -0.330002   
4               4  -2.413875  -1.050915  -0.031770   0.581324   

   cat__island_Biscoe  cat__island_Dream  cat__island_Torgersen species  split  
0                 0.0                0.0                    1.0  Adelie  train  
1                 0.0                0.0                    1.0  Adelie    val  
2                 0.0                0.0                    1.0  Adelie  train  
3                 0.0                0.0                    1.0  Adelie  train  
4                 0.0                0.0                    1.0  Adelie  train  


### Part 5 — Verify my Result

In [47]:
processed_feature_cols = list(feature_names)

print("Missing values:",
      processed_data[processed_feature_cols].isna().sum().sum())

print("All features numerical:",
      processed_data[processed_feature_cols].select_dtypes(include=np.number).shape[1]
      == len(processed_feature_cols))

Missing values: 0
All features numerical: True


In [48]:
original_features = data.drop(columns=["species", "split"]).shape[1]

processed_features_count = len(processed_feature_cols)

print("Original Features:", original_features)
print("Processed Features:", processed_features_count)

Original Features: 7
Processed Features: 7


In [49]:
print("PCA Components:", pca.n_components_)
print("Cumulative Explained Variance:",
      pca.explained_variance_ratio_.sum())

PCA Components: 4
Cumulative Explained Variance: 0.9812771158399247


In [50]:
print("Original Rows:", len(data))
print("Processed Rows:", len(processed_data))

print("\nOriginal Split Counts:")
print(data["split"].value_counts())

print("\nProcessed Split Counts:")
print(processed_data["split"].value_counts())

Original Rows: 344
Processed Rows: 344

Original Split Counts:
split
train    241
val       69
test      34
Name: count, dtype: int64

Processed Split Counts:
split
train    241
val       69
test      34
Name: count, dtype: int64
